Initial experiments with Logistic Regression

In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.sparse import csr_matrix
from tqdm import tqdm

def compute_transferability(orders_df, substitutes_df, product_info, output_csv=None, top_n=None):
    """
    Compute DTR (transferability %) for already identified substitutes.

    Parameters
    ----------
    orders_df : pd.DataFrame
        Must contain 'order_id', 'product_id', and scaled order-level features.
    substitutes_df : pd.DataFrame
        Already identified substitutes with columns:
        'product_id', 'substitute_id', 'score', 'rank', 'penetration_similarity',
        'substitution_index', 'conditional', 'jaccard'.
    product_info : pd.DataFrame
        Must contain 'product_id', 'aisle_id', 'department_id'.
    output_csv : str, optional
        CSV path to save results.
    top_n : int, optional
        Only use top N substitutes per product.
    """

    if output_csv:
        with open(output_csv, 'w') as f:
            f.write("product_id,substitute_id,score,rank,raw_dtr,ame,adjusted_dtr,coef,p_value,pseudo_r2\n")

    # Precompute order × product matrix
    orders_df['order_idx'] = orders_df['order_id'].astype('category').cat.codes
    orders_df['product_idx'] = orders_df['product_id'].astype('category').cat.codes
    order_idx_map = dict(enumerate(orders_df['order_id'].astype('category').cat.categories))
    product_idx_map = dict(enumerate(orders_df['product_id'].astype('category').cat.categories))

    n_orders = orders_df['order_idx'].max() + 1
    n_products = orders_df['product_idx'].max() + 1

    data = np.ones(len(orders_df), dtype=np.uint8)
    order_product_matrix = csr_matrix((data, (orders_df['order_idx'], orders_df['product_idx'])),
                                     shape=(n_orders, n_products))

    # Unique order-level features
    order_features = orders_df[['order_idx', 'order_size_scaled', 
                                'orders_per_user_scaled', 'prod_per_user_scaled']]\
                                .drop_duplicates('order_idx').set_index('order_idx')

    for product_id, group in tqdm(substitutes_df.groupby('product_id'), desc="Processing products"):
        if top_n:
            group = group.nsmallest(top_n, 'rank')

        if product_id not in product_idx_map.values():
            continue
        idx_A = {v:k for k,v in product_idx_map.items()}[product_id]
        orders_A_idx = set(order_product_matrix[:, idx_A].nonzero()[0])
        if not orders_A_idx:
            continue

        # Temporary list to hold all adjusted DTRs for this product
        results_list = []

        for _, row in group.iterrows():
            B = row['substitute_id']
            if B not in product_idx_map.values():
                continue
            idx_B = {v:k for k,v in product_idx_map.items()}[B]
            orders_B_idx = set(order_product_matrix[:, idx_B].nonzero()[0])

            # Raw DTR = fraction of A's orders that include B
            raw_dtr = len(orders_B_idx & orders_A_idx) / len(orders_A_idx) if orders_A_idx else 0

            # Construct per-order feature matrix for logistic regression
            X = order_features.copy()
            X['A_absent'] = (~X.index.isin(orders_A_idx)).astype(int)
            X['pen_similarity'] = row['penetration_similarity']
            X['score'] = row['score']
            X['substitution_index'] = row['substitution_index']
            X['conditional'] = row['conditional']
            X['jaccard'] = row['jaccard']
            X = sm.add_constant(X)

            y = X.index.isin(orders_B_idx).astype(int)

            try:
                logit_model = sm.Logit(y, X).fit(disp=0)
            except Exception as e:
                print(f"Error fitting Logit model for {product_id}-{B}: {e}")
                continue

            # Compute AME using absolute value
            X_present = X.copy(); X_present['A_absent'] = 0
            X_absent = X.copy(); X_absent['A_absent'] = 1
            pred_present = logit_model.predict(X_present)
            pred_absent = logit_model.predict(X_absent)
            ame = abs((pred_absent - pred_present).mean())  # use absolute value

            adjusted_dtr = raw_dtr * ame

            results_list.append({
                'product_id': product_id,
                'substitute_id': B,
                'score': row['score'],
                'rank': row['rank'],
                'raw_dtr': raw_dtr,
                'ame': ame,
                'adjusted_dtr': adjusted_dtr,
                'coef': logit_model.params['A_absent'],
                'p_value': logit_model.pvalues['A_absent'],
                'pseudo_r2': 1 - (logit_model.llf / logit_model.llnull),
            })

        # Normalize adjusted DTRs for this product so sum <= 1
        if results_list:
            total_adj_dtr = sum(r['adjusted_dtr'] for r in results_list)
            if total_adj_dtr > 1:
                for r in results_list:
                    r['adjusted_dtr'] /= total_adj_dtr

            # Save to CSV
            if output_csv:
                pd.DataFrame(results_list).to_csv(output_csv, mode='a', index=False, header=False)
            else:
                return results_list  # return all substitutes for this product

orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')
order_features_df = pd.read_csv('../data/cleaned/order-features.csv')  
orders_df = orders_full_df.merge(order_features_df, on='order_id', how='left')

product_info_df = pd.read_csv('../data/cleaned/product-features.csv')   

subs_df = pd.read_csv('../data/validation/substitutes/final-subs.csv')
chosen_ids = subs_df['product_id'].drop_duplicates().sample(5, random_state=42)
substitutes_df = subs_df[subs_df['product_id'].isin(chosen_ids)]
  

compute_transferability(orders_df, 
    substitutes_df, 
    product_info_df, 
    output_csv="../data/results/obj2/log-transfer5.csv",
    top_n=10
    )

Processing products:   0%|          | 0/5 [00:00<?, ?it/s]

Error fitting Logit model for 1146-23537: Singular matrix


Processing products:  20%|██        | 1/5 [03:51<15:25, 231.29s/it]

Error fitting Logit model for 4818-39461: Singular matrix
Error fitting Logit model for 4818-20066: Singular matrix


Processing products:  40%|████      | 2/5 [07:03<10:24, 208.12s/it]

Error fitting Logit model for 4818-24512: Singular matrix
Error fitting Logit model for 8817-19488: Singular matrix
Error fitting Logit model for 8817-19488: Singular matrix


Processing products:  80%|████████  | 4/5 [14:30<03:39, 219.17s/it]

Error fitting Logit model for 34628-13740: Singular matrix


Processing products: 100%|██████████| 5/5 [18:06<00:00, 217.33s/it]
